# Crop Disease Detection
### Fine-tuned MobileNetV2 Feature Extraction + XGBoost Classification

**Pipeline overview (sequential):**
1. Scan PlantVillage dataset → stratified 70/15/15 train/val/test split
2. Fine-tune MobileNetV2 top layers on plant disease images (lr=1e-4, EarlyStopping)
3. Strip Dense head → save `finetuned_feature_extractor.keras`
4. Re-extract 1,280-dim features from all splits using fine-tuned extractor → `.npz` caches
5. Train XGBoost with inverse-frequency sample weights + Optuna hyperparameter search
6. Evaluate on held-out test set → metrics, confusion matrix, per-class F1
7. LangGraph confidence-routing inference demo

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import tensorflow as tf
import xgboost as xgb
import sklearn

import config
from src.utils import setup_output_dirs, get_logger

setup_output_dirs()
logger = get_logger("notebook")

print(f"TensorFlow   {tf.__version__}")
print(f"XGBoost      {xgb.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"Feature source : {config.FEATURE_SOURCE}")
print("All imports OK")

## 1. Data Pipeline
Scan PlantVillage folder → build class index → stratified 70/15/15 split.

In [ ]:
from src.data_pipeline import build_class_index, scan_dataset, stratified_split

print(f"Dataset path: {config.DATA_DIR}")

class_index = build_class_index(config.DATA_DIR)

print(f"\nClasses found ({len(class_index)}):")
for name, idx in class_index.items():
    folder = config.DATA_DIR / name
    count = sum(1 for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"  {idx:2d}  {name:<50} ({count:4d} images)")

df = scan_dataset(config.DATA_DIR, class_index)
counts = df.groupby('label_int').size()
print(f"\nTotal images : {len(df):,}")
print(f"Imbalance    : {counts.max()/counts.min():.1f}x  "
      f"(min={counts.min()}, max={counts.max()})")

In [ ]:
train_df, val_df, test_df = stratified_split(df)

print("Stratified split (70 / 15 / 15):")
print(f"  Train : {len(train_df):6,} images")
print(f"  Val   : {len(val_df):6,} images")
print(f"  Test  : {len(test_df):6,} images")
print(f"\nAll {len(class_index)} classes present in every split: "
      f"{all(s['label_int'].nunique()==len(class_index) for s in [train_df,val_df,test_df])}")

## 2. Augmentation & Preprocessing

**Training split** — 6-step augmentation pipeline applied before MobileNetV2:

| Step | Transform | Purpose |
|------|-----------|----------|
| 1 | RandomFlip horizontal | Handles left/right camera orientation |
| 2 | RandomRotation ±54° | Different phone angles |
| 3 | RandomZoom ±10% | Different distances from leaf |
| 4 | RandomBrightness ±20% | Lighting conditions |
| 5 | RandomContrast ±20% | Image quality variation |
| 6 | GaussianNoise σ=0.05 | Applied **after** preprocess_input — prevents memorising exact pixels |

**Val / Test splits** — only `preprocess_input` (pixel/127.5−1 → [−1,1]). No randomness.

> During fine-tuning, `preprocess_input` is baked **inside** the model, so it must **not** be applied externally during inference.

In [ ]:
from src.data_pipeline import build_augmentation_pipeline, build_preprocessing_pipeline

aug_pipeline = build_augmentation_pipeline()
pre_pipeline = build_preprocessing_pipeline()

print("Training augmentation pipeline:")
for i, layer in enumerate(aug_pipeline.layers):
    print(f"  {i+1}. {layer.name}")

print("\nVal/Test preprocessing pipeline:")
for i, layer in enumerate(pre_pipeline.layers):
    print(f"  {i+1}. {layer.name}")

## 3. Fine-Tuning MobileNetV2

Unfreezes top layers (index 100–153) and trains them on plant disease images with a small learning rate.  
After training the Dense head is stripped — only `Input → preprocess_input → Backbone → GAP` is kept as the feature extractor.

**Run once via:**
```bash
python scripts/finetune.py --unfreeze-from 100 --epochs 10 --lr 1e-4
```
Saves: `outputs/models/finetuned_feature_extractor.keras` + three `.npz` feature caches.

In [ ]:
extractor_path = config.FINETUNED_EXTRACTOR_PATH

if extractor_path.exists():
    print(f"Fine-tuned extractor found: {extractor_path}")
    print(f"Size: {extractor_path.stat().st_size / 1e6:.1f} MB")
    feature_extractor = tf.keras.models.load_model(str(extractor_path))
    print(f"Input  shape : {feature_extractor.input_shape}   (raw [0,255] pixels)")
    print(f"Output shape : {feature_extractor.output_shape}  (1280-dim feature vector)")
    print(f"Trainable params : {sum(tf.size(v).numpy() for v in feature_extractor.trainable_variables):,}")
else:
    print("Fine-tuned extractor not found.")
    print("Run:  python scripts/finetune.py")
    print("Then re-run this cell.")

In [ ]:
# Show MobileNetV2 layer freeze/trainable summary
if extractor_path.exists():
    trainable   = [l for l in feature_extractor.layers if l.trainable]
    frozen      = [l for l in feature_extractor.layers if not l.trainable]
    print(f"Total layers    : {len(feature_extractor.layers)}")
    print(f"Frozen  (0–99)  : {len(frozen)} layers")
    print(f"Trainable(100+) : {len(trainable)} layers")
    print(f"\nNote: preprocess_input is the FIRST operation inside this model.")
    print(f"      Pass raw [0,255] pixels at inference — do NOT pre-scale externally.")

## 4. Feature Extraction

All three splits are passed through the **fine-tuned** extractor to produce 1,280-dim feature vectors.  
Caches are saved once to `.npz`; all subsequent training runs load from disk in seconds.

- **Train** — augmented images → extractor → `train_features_finetuned.npz`
- **Val**   — preprocess only → extractor → `val_features_finetuned.npz`
- **Test**  — preprocess only → extractor → `test_features_finetuned.npz`

In [ ]:
from src.feature_extractor import features_cached, load_features, save_features, extract_features
from src.data_pipeline import image_generator, num_batches
from src.utils import build_label_encoder, save_label_encoder, load_label_encoder

all_cached = all(
    features_cached(p)
    for p in [config.TRAIN_FEATURES_PATH, config.VAL_FEATURES_PATH, config.TEST_FEATURES_PATH]
)

if all_cached:
    print("Feature caches found — loading from disk.")
    print(f"  {config.TRAIN_FEATURES_PATH.name}")
    print(f"  {config.VAL_FEATURES_PATH.name}")
    print(f"  {config.TEST_FEATURES_PATH.name}")
    X_train, y_train = load_features(config.TRAIN_FEATURES_PATH)
    X_val,   y_val   = load_features(config.VAL_FEATURES_PATH)
    X_test,  y_test  = load_features(config.TEST_FEATURES_PATH)
else:
    if not extractor_path.exists():
        raise RuntimeError("Run scripts/finetune.py first to create the feature caches.")
    print("Extracting features using fine-tuned extractor...")
    print("Note: fine-tuned model handles preprocess_input internally — pass raw pixels.")

    for split_df, path, desc in [
        (train_df, config.TRAIN_FEATURES_PATH, "Train"),
        (val_df,   config.VAL_FEATURES_PATH,   "Val"),
        (test_df,  config.TEST_FEATURES_PATH,  "Test"),
    ]:
        gen = image_generator(
            split_df["filepath"].tolist(), split_df["label_int"].values,
            batch_size=config.BATCH_SIZE,
            preprocess_fn=None,   # model applies preprocess_input internally
        )
        X, y = extract_features(feature_extractor, gen, num_batches(len(split_df)), desc=desc)
        save_features(X, y, path)

    X_train, y_train = load_features(config.TRAIN_FEATURES_PATH)
    X_val,   y_val   = load_features(config.VAL_FEATURES_PATH)
    X_test,  y_test  = load_features(config.TEST_FEATURES_PATH)

    label_names_sorted = sorted(class_index.keys())
    le = build_label_encoder(label_names_sorted)
    save_label_encoder(le, config.LABEL_ENCODER_PATH)

print(f"\nFeature shapes:")
print(f"  X_train : {X_train.shape}    y_train : {y_train.shape}")
print(f"  X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"  X_test  : {X_test.shape}     y_test  : {y_test.shape}")

le = load_label_encoder(config.LABEL_ENCODER_PATH)
label_names = list(le.classes_)
print(f"  Classes : {len(label_names)}")

## 5. Class Distribution

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

short_names = []
for n in label_names:
    if "___" in n:
        short_names.append(n.split("___")[-1].replace("_", " "))
    elif "__" in n:
        short_names.append(n.split("__")[-1].replace("_", " "))
    else:
        short_names.append(n.split("_", 1)[-1].replace("_", " "))

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#e74c3c' if c < 300 else '#f39c12' if c < 1000 else '#2ecc71' for c in counts]
ax.bar(range(len(unique)), counts, color=colors, edgecolor='white')
ax.set_xticks(range(len(unique)))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Training samples")
ax.set_title("Class Distribution in Training Split\nRed=minority (<300), Orange=moderate, Green=majority")
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / "class_distribution.png", dpi=150)
plt.show()

print(f"Imbalance ratio : {counts.max()}/{counts.min()} = {counts.max()/counts.min():.1f}x")
print(f"Sample weights (inverse-frequency) compensate — no SMOTE needed.")

## 6. XGBoost Training

**No SMOTE** — inverse-frequency sample weights handle the 21× class imbalance directly:  
`weight = N_total / (n_classes × class_count)` → rare classes get up to 9× higher gradient signal.

Three phases:
1. **Baseline** — fixed hyperparameters from `config.XGB_BASE_PARAMS`
2. **Optuna tuning** — 50 trials × 3-fold CV, maximises macro F1
3. **Retrain** — best params applied on full training set; params auto-saved back to `config.py`

In [ ]:
from src.classifier import compute_sample_weights, train_baseline

sample_weights = compute_sample_weights(y_train)
print(f"Sample weights — min: {sample_weights.min():.3f}  max: {sample_weights.max():.3f}  "
      f"ratio: {sample_weights.max()/sample_weights.min():.1f}x")
print(f"(Rare classes receive proportionally stronger gradient signal)")

print(f"\nTraining baseline XGBoost...")
baseline_model = train_baseline(X_train, y_train, X_val, y_val)

In [ ]:
from src.classifier import run_tuning

# Optuna: 50 trials × 3-fold stratified CV — TPE sampler + MedianPruner
study = run_tuning(X_train, y_train, n_trials=config.OPTUNA_N_TRIALS)

In [ ]:
from src.classifier import retrain_best, save_model
from src.utils import update_config_xgb_params

# Auto-save best params back to config.py for next run
updated = update_config_xgb_params(study.best_params)
print("config.py XGB_BASE_PARAMS updated:")
for k, v in updated.items():
    print(f"  {k} = {v}")

print("\nRetraining with best Optuna params on full training set...")
final_model = retrain_best(X_train, y_train, X_val, y_val, study.best_params)
save_model(final_model, config.XGB_MODEL_PATH)
print(f"Model saved → {config.XGB_MODEL_PATH.relative_to(PROJECT_ROOT)}")

## 7. Optuna Trial History

In [ ]:
from src.evaluate import plot_optuna_history

plot_optuna_history(study, config.PLOTS_DIR / "optuna_history.png")

img = plt.imread(str(config.PLOTS_DIR / "optuna_history.png"))
plt.figure(figsize=(10, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f"Best trial #{study.best_trial.number}  →  macro F1 = {study.best_value:.4f}")

## 8. Test Set Evaluation

> **TEST split is evaluated exactly ONCE** — it was never seen during fine-tuning, feature extraction, XGBoost training, or Optuna tuning. This gives an unbiased performance estimate.

In [ ]:
from src.evaluate import compute_all_metrics, save_classification_report

metrics = compute_all_metrics(final_model, X_test, y_test, label_names)

print("=" * 52)
print("TEST SET RESULTS  (3,097 held-out images)")
print("=" * 52)
print(f"  Accuracy       : {metrics['accuracy']*100:.2f}%")
print(f"  Macro F1       : {metrics['macro_f1']:.4f}")
print(f"  Weighted F1    : {metrics['weighted_f1']:.4f}")
print(f"  Top-3 Accuracy : {metrics['top3_accuracy']*100:.2f}%")
print("=" * 52)

print(f"\n{'Class':<48} {'Prec':>5}  {'Rec':>5}  {'F1':>5}  {'N':>5}")
print("-" * 70)
for i, name in enumerate(label_names):
    pc = metrics['per_class'][i]
    print(f"  {name:<46} {pc['precision']:>5.2f}  {pc['recall']:>5.2f}  {pc['f1']:>5.2f}  {pc['support']:>5}")
print("-" * 70)
print(f"  {'macro avg':<46} {metrics['avg_precision']:>5.2f}  {metrics['avg_recall']:>5.2f}  {metrics['macro_f1']:>5.2f}  {len(y_test):>5}")

save_classification_report(final_model, X_test, y_test, label_names,
                           config.REPORTS_DIR / "classification_report.txt")

## 9. Confusion Matrix

In [ ]:
from src.evaluate import plot_confusion_matrix

y_pred = final_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred, label_names,
                      config.PLOTS_DIR / "confusion_matrix.png")

img = plt.imread(str(config.PLOTS_DIR / "confusion_matrix.png"))
plt.figure(figsize=(12, 10))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 10. Per-Class F1

In [ ]:
from src.evaluate import plot_per_class_f1

plot_per_class_f1(metrics["per_class"], label_names,
                  config.PLOTS_DIR / "per_class_f1.png")

img = plt.imread(str(config.PLOTS_DIR / "per_class_f1.png"))
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 11. XGBoost Validation Loss Curve

In [ ]:
evals = final_model.evals_result()
val_loss = evals.get("validation_0", {}).get("mlogloss", [])

if val_loss:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(val_loss, color='#e74c3c', linewidth=1.5, label='Val mlogloss')
    ax.set_xlabel("Number of trees")
    ax.set_ylabel("mlogloss")
    ax.set_title("XGBoost Validation Loss  (lower = better)")
    ax.legend()
    ax.grid(alpha=0.3)
    min_idx = int(np.argmin(val_loss))
    ax.axvline(x=min_idx, color='gray', linestyle='--', alpha=0.7)
    ax.annotate(f'Best: {val_loss[min_idx]:.4f} @ tree {min_idx}',
                xy=(min_idx, val_loss[min_idx]),
                xytext=(min_idx + 10, val_loss[min_idx] + 0.05),
                arrowprops=dict(arrowstyle='->'))
    plt.tight_layout()
    plt.savefig(config.PLOTS_DIR / "xgb_loss_curve.png", dpi=150)
    plt.show()
else:
    print("No evaluation log — model loaded from disk.")

## 12. LangGraph Inference Demo

The LangGraph workflow wraps inference in a stateful pipeline with confidence routing:

```
field photo
    → validate_image   (file exists? correct format?)
    → run_prediction   (extractor → 1280 → XGBoost → top-3)
    → confidence_router
          ≥ 60%  → final_response      (diagnosis + top-3)
          < 60%  → low_confidence      (ask for clearer photo)
```

Confidence threshold controlled by `config.CONFIDENCE_THRESHOLD = 0.60`.

In [ ]:
from src.langgraph_workflow import build_graph

# Build and compile the LangGraph app (done once — reusable)
app = build_graph()
print("LangGraph workflow compiled.")
print(f"Confidence threshold : {config.CONFIDENCE_THRESHOLD:.0%}")

In [ ]:
# Pick a random test image
rng = np.random.default_rng(99)
sample_idx  = rng.integers(0, len(test_df))
sample_path = test_df.iloc[sample_idx]["filepath"]
true_label  = test_df.iloc[sample_idx]["label_str"]

print(f"Sample image : {Path(sample_path).name}")
print(f"True label   : {true_label}")

initial_state = {
    "image_path" : sample_path,
    "status"     : "pending",
    "predictions": None,
    "confidence" : None,
    "message"    : None,
}

result = app.invoke(initial_state)

print("\n" + "=" * 52)
print(result["message"])
print("=" * 52)

top1_correct = (
    result["predictions"] is not None and
    result["predictions"][0]["raw_label"] == true_label
)
print(f"\nTrue label: {true_label}")
print(f"Result: {'✅ Correct (top-1)' if top1_correct else '❌ Wrong top-1'}")

In [ ]:
from PIL import Image as PILImage

preds = result["predictions"]

if preds:
    img = PILImage.open(sample_path).convert("RGB")

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 5))

    ax_img.imshow(img)
    ax_img.set_title(f"Input image\nTrue: {true_label}", fontsize=9)
    ax_img.axis('off')

    labels = [f"#{p['rank']}  {p['crop']} — {p['disease']}" for p in preds]
    confs  = [p['confidence'] * 100 for p in preds]
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    bars   = ax_bar.barh(range(len(preds)), confs, color=colors)
    ax_bar.set_yticks(range(len(preds)))
    ax_bar.set_yticklabels(labels, fontsize=9)
    ax_bar.set_xlabel("Confidence (%)")
    ax_bar.set_title("LangGraph Top-3 Predictions")
    ax_bar.set_xlim(0, 105)
    ax_bar.axvline(x=config.CONFIDENCE_THRESHOLD * 100, color='gray',
                   linestyle='--', alpha=0.7, label=f"Threshold {config.CONFIDENCE_THRESHOLD:.0%}")
    ax_bar.legend(fontsize=8)
    for bar, conf in zip(bars, confs):
        ax_bar.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                    f"{conf:.1f}%", va='center', fontsize=9)
    ax_bar.invert_yaxis()

    plt.tight_layout()
    plt.savefig(config.PLOTS_DIR / "inference_demo.png", dpi=150)
    plt.show()

## Summary

| Metric | Value |
|--------|-------|
| Test Accuracy | — |
| Macro F1 | — |
| Weighted F1 | — |
| **Top-3 Accuracy** | **—** |
| Feature source | `finetuned_feature_extractor.keras` |
| Preprocessing | Inverse-frequency sample weights (no SMOTE) |
| Inference | LangGraph confidence routing (threshold 60%) |

*(Run all cells to populate the metrics above)*

**Key design decisions:**
- **Fine-tuning MobileNetV2** (layers 100–153) on plant disease images improves feature quality vs frozen backbone
- **No SMOTE** — SMOTE blurs class boundaries in 1280-dim feature space and is redundant with sample weights
- **Inverse-frequency sample weights** — rare classes (e.g. Potato_healthy with 106 images) receive ~9× higher gradient signal
- **LangGraph confidence router** — low-confidence predictions (< 60%) trigger a tip asking for a clearer photo rather than returning an unreliable diagnosis
- **Auto-config update** — Optuna best params are written back to `config.py` after each run so the next baseline already starts from the best known values